# Mini-Project: Soccer Results Analytics
## Part 5 - Machine Learning with Spark MLlib
**Objectif:** Prédire le résultat d'un match (home_win/draw/away_win)
à partir des caractéristiques des équipes.

**Training set:** Données historiques 2007-2023

**Test set:** Saison 2024/2025 (données scrapées Wikipedia)

**Auteur:** Raghye Meissara Bilal

## SparkSession

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("SoccerMLModel") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f" Spark {spark.version} prêt!")

 Spark 3.5.3 prêt!


## Charger les données

In [2]:
# Charger données historiques préparées
df = spark.read.parquet("/workspace/data/outputs/prepared")

print(f"Total matchs: {df.count()}")
print("\nDistribution des résultats:")
df.groupBy("result").count().orderBy("count", ascending=False).show()

# Train set: 2007-2023
train_df = df.filter(col("match_year") < 2024)

# Test set: 2024 (saison actuelle)
test_df = df.filter(col("match_year") == 2024)

print(f"\nTrain set: {train_df.count()} matchs (2007-2023)")
print(f"Test set : {test_df.count()} matchs (2024)")

Total matchs: 2744

Distribution des résultats:
+--------+-----+
|  result|count|
+--------+-----+
|home_win| 1086|
|away_win|  901|
|    draw|  757|
+--------+-----+


Train set: 2504 matchs (2007-2023)
Test set : 240 matchs (2024)


## Feature Engineering

In [3]:
from pyspark.ml.feature import StringIndexer, VectorAssembler, OneHotEncoder
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Indexer les équipes
home_indexer = StringIndexer(inputCol="home_team", outputCol="home_team_idx", handleInvalid="keep")
away_indexer = StringIndexer(inputCol="away_team", outputCol="away_team_idx", handleInvalid="keep")

# Indexer le label (résultat)
label_indexer = StringIndexer(inputCol="result", outputCol="label", handleInvalid="keep")

# Assembler les features
assembler = VectorAssembler(
    inputCols=["home_team_idx", "away_team_idx", "match_year"],
    outputCol="features"
)

print(" Feature engineering défini!")
print("Features utilisées:")
print("  - home_team_idx : équipe domicile (encodée)")
print("  - away_team_idx : équipe extérieur (encodée)")
print("  - match_year    : année du match")
print("  - label         : résultat (home_win=0, draw=1, away_win=2)")

 Feature engineering défini!
Features utilisées:
  - home_team_idx : équipe domicile (encodée)
  - away_team_idx : équipe extérieur (encodée)
  - match_year    : année du match
  - label         : résultat (home_win=0, draw=1, away_win=2)


## Entraîner les modèles

In [6]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,
    maxDepth=5,
    maxBins=64  
)

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=20
)

pipeline_rf = Pipeline(stages=[
    home_indexer, away_indexer, label_indexer, assembler, rf
])
pipeline_lr = Pipeline(stages=[
    home_indexer, away_indexer, label_indexer, assembler, lr
])

print(" Entraînement Random Forest...")
model_rf = pipeline_rf.fit(train_df)
print(" Random Forest entraîné!")

print(" Entraînement Logistic Regression...")
model_lr = pipeline_lr.fit(train_df)
print(" Logistic Regression entraîné!")

 Entraînement Random Forest...
 Random Forest entraîné!
 Entraînement Logistic Regression...
 Logistic Regression entraîné!


## Évaluation sur le test set

In [7]:
# Évaluer sur le test set (saison 2024)
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

pred_rf = model_rf.transform(test_df)
pred_lr = model_lr.transform(test_df)

acc_rf = evaluator.evaluate(pred_rf)
acc_lr = evaluator.evaluate(pred_lr)

# F1-score
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

f1_rf = evaluator_f1.evaluate(pred_rf)
f1_lr = evaluator_f1.evaluate(pred_lr)

print("=== Résultats sur Test Set (2024) ===")
print(f"Random Forest      - Accuracy: {acc_rf:.4f} ({acc_rf*100:.1f}%) | F1: {f1_rf:.4f}")
print(f"Logistic Regression- Accuracy: {acc_lr:.4f} ({acc_lr*100:.1f}%) | F1: {f1_lr:.4f}")
print(f"\n Meilleur modèle: {'Random Forest' if acc_rf >= acc_lr else 'Logistic Regression'}")

=== Résultats sur Test Set (2024) ===
Random Forest      - Accuracy: 0.4042 (40.4%) | F1: 0.3038
Logistic Regression- Accuracy: 0.3833 (38.3%) | F1: 0.3137

 Meilleur modèle: Random Forest


## Inspecter les prédictions

In [8]:
from pyspark.ml.feature import IndexToString

# Convertir les prédictions en labels lisibles
label_converter = IndexToString(
    inputCol="prediction",
    outputCol="predicted_result",
    labels=model_rf.stages[2].labels
)

pred_rf_readable = label_converter.transform(pred_rf)

print("=== Exemples de prédictions ===")
pred_rf_readable.select(
    "home_team", "away_team",
    "home_goals", "away_goals",
    "result", "predicted_result"
).show(20, truncate=False)

# Analyse des erreurs
print("\n=== Analyse des erreurs ===")
errors = pred_rf_readable.filter(col("result") != col("predicted_result"))
print(f"Total erreurs: {errors.count()}/{pred_rf_readable.count()}")

print("\nDistribution des prédictions:")
pred_rf_readable.groupBy("predicted_result").count().show()

print("\nDistribution réelle:")
pred_rf_readable.groupBy("result").count().show()

=== Exemples de prédictions ===
+-----------------+-----------------+----------+----------+--------+----------------+
|home_team        |away_team        |home_goals|away_goals|result  |predicted_result|
+-----------------+-----------------+----------+----------+--------+----------------+
|Garde Nationale  |Nouakchott King's|1         |1         |draw    |home_win        |
|Toulde           |Ksar             |3         |0         |home_win|home_win        |
|Kaedi            |Douane           |2         |0         |home_win|home_win        |
|Gendrim          |Inter Nouakchott |0         |0         |draw    |home_win        |
|SNIM             |Al-Hilal Omdurman|0         |1         |away_win|home_win        |
|Nouadhibou       |Nzeidane         |2         |0         |home_win|home_win        |
|Pompiers         |Chemal           |1         |2         |away_win|away_win        |
|Tevragh-Zeina    |Al-Merreikh      |4         |0         |home_win|home_win        |
|Gendrim          |Al-

## Sauvegarder le meilleur modèle

In [10]:
# Sauvegarder le meilleur modèle
best_model = model_rf if acc_rf >= acc_lr else model_lr
model_path = "/workspace/data/outputs/models/soccer_result_model"

best_model.write().overwrite().save(model_path)
print(f" Meilleur modèle sauvegardé!")
print(f"   Path: {model_path}")
print(f"   Modèle: {'Random Forest' if acc_rf >= acc_lr else 'Logistic Regression'}")
best_acc = acc_rf if acc_rf >= acc_lr else acc_lr
print(f"   Accuracy: {best_acc*100:.1f}%")

 Meilleur modèle sauvegardé!
   Path: /workspace/data/outputs/models/soccer_result_model
   Modèle: Random Forest
   Accuracy: 40.4%


## Discussion limitations

In [11]:
print("=== Discussion du modèle ===")
print("""
Accuracy: 40.4% (Random Forest) vs baseline ~40% (toujours home_win)

Limitations:
1. Features limitées: seulement équipe + année
   → Pas d'historique récent, forme des équipes, blessures

2. Déséquilibre des classes:
   → home_win=40%, away_win=33%, draw=27%
   → Le modèle biaisé vers home_win

3. Nouvelles équipes en 2024:
   → Al-Hilal Omdurman non présent dans training set
   → handleInvalid='keep' mais sans historique

Améliorations possibles:
1. Ajouter features: forme récente (5 derniers matchs)
2. Utiliser le classement actuel comme feature
3. Rééchantillonner les classes (SMOTE)
4. Gradient Boosting (GBT)
""")

=== Discussion du modèle ===

Accuracy: 40.4% (Random Forest) vs baseline ~40% (toujours home_win)

Limitations:
1. Features limitées: seulement équipe + année
   → Pas d'historique récent, forme des équipes, blessures

2. Déséquilibre des classes:
   → home_win=40%, away_win=33%, draw=27%
   → Le modèle biaisé vers home_win

3. Nouvelles équipes en 2024:
   → Al-Hilal Omdurman non présent dans training set
   → handleInvalid='keep' mais sans historique

Améliorations possibles:
1. Ajouter features: forme récente (5 derniers matchs)
2. Utiliser le classement actuel comme feature
3. Rééchantillonner les classes (SMOTE)
4. Gradient Boosting (GBT)



## Amélioration 1 : Features enrichies

In [12]:
from pyspark.sql import Window
from pyspark.sql.functions import *

# Calculer les stats historiques par équipe
# (win rate, avg goals scored, avg goals conceded)
home_stats = df.groupBy("home_team").agg(
    avg(when(col("result") == "home_win", 1).otherwise(0)).alias("home_win_rate"),
    avg("home_goals").alias("avg_home_goals_scored"),
    avg("away_goals").alias("avg_home_goals_conceded")
).withColumnRenamed("home_team", "team")

away_stats = df.groupBy("away_team").agg(
    avg(when(col("result") == "away_win", 1).otherwise(0)).alias("away_win_rate"),
    avg("away_goals").alias("avg_away_goals_scored"),
    avg("home_goals").alias("avg_away_goals_conceded")
).withColumnRenamed("away_team", "team")

# Combiner les stats
team_stats = home_stats.join(away_stats, "team", "outer") \
    .fillna(0)

print(" Stats par équipe calculées!")
team_stats.show(5, truncate=False)

# Enrichir le dataset avec les stats
df_enriched = df \
    .join(team_stats.select(
        col("team").alias("home_team"),
        col("home_win_rate"),
        col("avg_home_goals_scored"),
        col("avg_home_goals_conceded")
    ), "home_team", "left") \
    .join(team_stats.select(
        col("team").alias("away_team"),
        col("away_win_rate"),
        col("avg_away_goals_scored"),
        col("avg_away_goals_conceded")
    ), "away_team", "left") \
    .fillna(0)

print(f"\n Dataset enrichi: {df_enriched.count()} matchs")
df_enriched.show(3, truncate=False)

 Stats par équipe calculées!
+-----------------+-------------------+---------------------+-----------------------+-------------------+---------------------+-----------------------+
|team             |home_win_rate      |avg_home_goals_scored|avg_home_goals_conceded|away_win_rate      |avg_away_goals_scored|avg_away_goals_conceded|
+-----------------+-------------------+---------------------+-----------------------+-------------------+---------------------+-----------------------+
|ASAC Concorde    |0.5549738219895288 |1.7277486910994764   |0.8952879581151832     |0.45595854922279794|1.4404145077720207   |0.9948186528497409     |
|ASC Entente      |0.15384615384615385|0.9230769230769231   |1.6153846153846154     |0.07692307692307693|0.6923076923076923   |1.6923076923076923     |
|Al-Hilal Omdurman|0.625              |1.8125               |0.5625                 |0.7142857142857143 |1.8571428571428572   |0.6428571428571429     |
|Al-Merreikh      |0.4                |1.3333333333333333  

## Amélioration 2 : Équilibrer les classes

In [14]:
from pyspark.ml.classification import (
    RandomForestClassifier,
    LogisticRegression,
    DecisionTreeClassifier
)
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Modèles (sans GBT car multi-classe non supporté)
rf_v2 = RandomForestClassifier(
    featuresCol="features", labelCol="label",
    numTrees=200, maxDepth=8, maxBins=64
)

dt = DecisionTreeClassifier(
    featuresCol="features", labelCol="label",
    maxDepth=8, maxBins=64
)

lr_v2 = LogisticRegression(
    featuresCol="features", labelCol="label",
    maxIter=50
)

stages_base = [home_indexer, away_indexer, label_indexer, assembler_v2]

pipeline_rf2 = Pipeline(stages=stages_base + [rf_v2])
pipeline_dt  = Pipeline(stages=stages_base + [dt])
pipeline_lr2 = Pipeline(stages=stages_base + [lr_v2])

print(" Entraînement Random Forest v2...")
model_rf2 = pipeline_rf2.fit(train_balanced)
print(" RF v2 entraîné!")

print(" Entraînement Decision Tree...")
model_dt = pipeline_dt.fit(train_balanced)
print(" Decision Tree entraîné!")

print(" Entraînement Logistic Regression v2...")
model_lr2 = pipeline_lr2.fit(train_balanced)
print(" LR v2 entraîné!")

 Entraînement Random Forest v2...
 RF v2 entraîné!
 Entraînement Decision Tree...
 Decision Tree entraîné!
 Entraînement Logistic Regression v2...
 LR v2 entraîné!


## Évaluation comparative

In [15]:
evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction",
    metricName="accuracy"
)
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction",
    metricName="f1"
)

pred_rf2 = model_rf2.transform(test_enriched)
pred_dt  = model_dt.transform(test_enriched)
pred_lr2 = model_lr2.transform(test_enriched)

acc_rf2 = evaluator.evaluate(pred_rf2)
acc_dt  = evaluator.evaluate(pred_dt)
acc_lr2 = evaluator.evaluate(pred_lr2)

f1_rf2 = evaluator_f1.evaluate(pred_rf2)
f1_dt  = evaluator_f1.evaluate(pred_dt)
f1_lr2 = evaluator_f1.evaluate(pred_lr2)

print("=== Comparaison des modèles ===")
print(f"{'Modèle':<30} {'Accuracy':>10} {'F1':>10}")
print("-" * 52)
print(f"{'RF v1 (baseline)':<30} {'40.4%':>10} {'0.30':>10}")
print(f"{'RF v2 (enrichi+équilibré)':<30} {acc_rf2*100:>9.1f}% {f1_rf2:>10.4f}")
print(f"{'Decision Tree':<30} {acc_dt*100:>9.1f}%  {f1_dt:>10.4f}")
print(f"{'LR v2':<30} {acc_lr2*100:>9.1f}%  {f1_lr2:>10.4f}")

# Distribution prédictions RF v2
from pyspark.ml.feature import IndexToString
label_converter = IndexToString(
    inputCol="prediction",
    outputCol="predicted_result",
    labels=model_rf2.stages[2].labels
)
pred_rf2_readable = label_converter.transform(pred_rf2)

print("\n=== Distribution prédictions RF v2 ===")
pred_rf2_readable.groupBy("predicted_result").count().show()
print("=== Distribution réelle ===")
pred_rf2_readable.groupBy("result").count().show()

# Sauvegarder le meilleur modèle
best_acc = acc_rf2
best_model_v2 = model_rf2
if acc_dt > best_acc:
    best_acc = acc_dt
    best_model_v2 = model_dt
if acc_lr2 > best_acc:
    best_acc = acc_lr2
    best_model_v2 = model_lr2

best_model_v2.write().overwrite().save(
    "/workspace/data/outputs/models/soccer_result_model_v2"
)
print(f"\n Meilleur modèle v2 sauvegardé! Accuracy: {best_acc*100:.1f}%")

=== Comparaison des modèles ===
Modèle                           Accuracy         F1
----------------------------------------------------
RF v1 (baseline)                    40.4%       0.30
RF v2 (enrichi+équilibré)           46.7%     0.4703
Decision Tree                       47.9%      0.4794
LR v2                               53.3%      0.5365

=== Distribution prédictions RF v2 ===
+----------------+-----+
|predicted_result|count|
+----------------+-----+
|        away_win|   76|
|            draw|   81|
|        home_win|   83|
+----------------+-----+

=== Distribution réelle ===
+--------+-----+
|  result|count|
+--------+-----+
|away_win|   68|
|    draw|   74|
|home_win|   98|
+--------+-----+


 Meilleur modèle v2 sauvegardé! Accuracy: 53.3%
